<a href="https://colab.research.google.com/github/marcoslund/ViT-for-101-food-app/blob/feat%2Fdeit/deit/notebooks/3.3-deit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.3 — DeiT-tiny fine-tuning completo — Food-101

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoslund/ViT-for-101-food-app/blob/feat/deit/notebooks/3.3-deit.ipynb)

Entrenamiento completo de **DeiT-tiny** (`facebook/deit-tiny-patch16-224`) con la estructura del proyecto y los artefactos de `2.0-preprocessing.ipynb`.

DeiT (*Data-efficient image Transformer*) es un **ViT** entrenado de forma eficiente en ImageNet-1k con augmentation fuerte y **destilación**; misma arquitectura que ViT (parches + encoder transformer, sin ventanas jerárquicas), pero diminuto (~5.7M params). Teoría: DeiT aparece en la Clase 1 y la destilación en las Clases 5-6 del material CEIA-ViT.

- El **código de modelo y entrenamiento vive en `modeling/training.py`**; acá solo se llaman funciones.
- Reusa `config.py` y `preprocessing/loaders.py`: no se reimplementan resize, crop, normalización ni augmentation.
- La evaluación usa `modeling/evaluation.py`, **el mismo código que MobileViT, ViT y Swin**.
- `test` se usa una sola vez al final; la pregunta del proyecto la contesta el **subset del benchmark por tercil**.
- Al terminar, guarda el mejor modelo y lo **descarga a tu computadora** (§8.1).
- Checkpoints bajo `models/deit/full/`; resultados livianos en `reports/results/deit/`.

**Por qué DeiT-tiny:** es el modelo más liviano del benchmark con checkpoint limpio. Da la escala más chica de la familia ViT: **DeiT-tiny (~5.7M) → Swin-tiny (~28M) → ViT-base (~86M)**, más MobileViT como especialista on-device. En una T4 entrena en pocas horas.

## 0. Antes de ejecutar

Desde la raíz del repo:

```bash
uv sync --extra deep
uv run jupyter lab
```

El preprocessing debe existir (`make preprocess` o `notebooks/2.0-preprocessing.ipynb`). En **Colab** no hace falta nada previo: §0.2 y §0.3 arman todo.

## 0.1 Parámetros de ejecución

- `USE_DRIVE` (solo Colab): guarda dataset, checkpoints y resultados en Google Drive. El disco de Colab se borra al desconectarse; sin Drive, una desconexión a mitad de entrenamiento pierde todo.
- `RESUME`: `True` retoma desde el último checkpoint de `OUTPUT_DIR` (corrida cortada). Con `False` y checkpoints viejos, la notebook se detiene en vez de mezclarlos.
- `REPO_BRANCH`: rama que Colab clona. Mientras DeiT no esté en `main`, tiene que ser la rama que trae `deit` en el registry y `evaluation.py`/`training.py`.

In [1]:
# Solo Colab: persistir dataset, checkpoints y resultados en Google Drive.
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/ceia-vpc3"

# True = retomar desde el ultimo checkpoint de OUTPUT_DIR (corrida cortada).
RESUME = False

# Rama a clonar en Colab (una vez mergeado a main, poner "main").
REPO_BRANCH = "feat/deit"

## 0.2 Entorno (Colab)

En Colab clona el repo (rama `REPO_BRANCH`) e instala el paquete con el extra `deep`. `torch` ya viene con CUDA en Colab. Localmente solo verifica que el paquete esté instalado.

In [2]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"Colab: {IN_COLAB}")


def _paquete_disponible():
    try:
        import vit_for_101_food_app  # noqa: F401

        return True
    except ImportError:
        return False


if IN_COLAB:
    REPO_URL = "https://github.com/marcoslund/ViT-for-101-food-app.git"
    REPO_DIR = "/content/ViT-for-101-food-app"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        # Ya clonado: traer la ultima version de la rama. Si actualizo codigo del paquete,
        # despues hay que reiniciar el runtime para que se reimporte.
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
    sys.path.insert(0, REPO_DIR)

    if not _paquete_disponible():
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[deep]"], check=True
        )
    os.chdir(f"{REPO_DIR}/notebooks")

    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
elif not _paquete_disponible():
    raise RuntimeError(
        "vit_for_101_food_app no esta instalado en este kernel. "
        "Elegi el kernel del .venv del proyecto (ver 'Antes de ejecutar')."
    )

print("paquete disponible:", _paquete_disponible())

Colab: True
Mounted at /content/drive


2026-09-25 21:59:54.144 | INFO     | vit_for_101_food_app.config:<module>:11 - PROJ_ROOT path is: /content/ViT-for-101-food-app


paquete disponible: True


## 0.3 Datos (Colab)

Lo que localmente hace `make preprocess`: descarga (o copia de Drive) el dataset, genera el split determinístico si falta y arma el cache. Localmente esta celda no hace nada.

In [3]:
if IN_COLAB:
    import shutil

    from vit_for_101_food_app import config
    from vit_for_101_food_app import dataset as prep

    tar_local = config.RAW_DATA_DIR / "food-101.tar.gz"
    tar_drive = f"{DRIVE_DIR}/food-101.tar.gz"
    config.RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    if USE_DRIVE and os.path.exists(tar_drive) and not tar_local.exists():
        print("copiando el dataset desde Drive...")
        shutil.copy(tar_drive, tar_local)

    prep.download()  # descarga (~5 GB) y extrae; idempotente
    if USE_DRIVE and not os.path.exists(tar_drive):
        print("guardando el dataset en Drive para la proxima sesion...")
        shutil.copy(tar_local, tar_drive)

    if not config.TRAIN_VAL_SPLIT.exists():
        prep.split()
    prep.cache(workers=os.cpu_count() or 2)  # idempotente

copiando el dataset desde Drive...
2026-09-25 22:02:29.276 | INFO     | vit_for_101_food_app.preprocessing.raw:ensure_dataset:105 - extrayendo en /content/ViT-for-101-food-app/data/raw
2026-09-25 22:03:58.020 | SUCCESS  | vit_for_101_food_app.preprocessing.raw:ensure_dataset:120 - Food-101 disponible en /content/ViT-for-101-food-app/data/raw/food-101
2026-09-25 22:03:58.983 | SUCCESS  | vit_for_101_food_app.dataset:download:37 - 4.77 GB en /content/ViT-for-101-food-app/data/raw/food-101
2026-09-25 22:03:59.659 | SUCCESS  | vit_for_101_food_app.preprocessing.splits:write_artifacts:112 - split escrito: 68175 train / 7575 val en 101 clases
sha256: 8f9fc5cb61748c625b746a91ad7276f0f267d5815aaaadabf30645e2c8348210


cache: 100%|██████████| 101000/101000 [09:28<00:00, 177.59it/s]


2026-09-25 22:13:33.217 | SUCCESS  | vit_for_101_food_app.preprocessing.cache:build_cache:150 - cache: 101000 escritas, 0 salteadas, 0 fallidas, 0 .tmp limpios en /content/ViT-for-101-food-app/data/interim/food-101-288
101000 imagenes, 3417 MB


In [4]:
from pathlib import Path
import sys

import pandas as pd
import torch
from sklearn.metrics import classification_report

from vit_for_101_food_app import config
from vit_for_101_food_app.modeling import evaluation, training
from vit_for_101_food_app.preprocessing import cache, loaders, processors, splits

MODEL_KEY = "deit"

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
else:
    print("Dispositivo: CPU (el full training en CPU es lento; usa una GPU).")

Python: 3.13.15
PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4
VRAM: 14.6 GB


## 1. Verificar artefactos de preprocessing

In [5]:
required_files = [
    config.TRAIN_VAL_SPLIT,
    config.TRAIN_VAL_MANIFEST,
    config.LABEL_MAP,
    config.CACHE_DIR / "cache_manifest.json",
]

missing = [Path(p) for p in required_files if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Faltan artefactos del preprocessing:\n"
        + "\n".join(f" - {p}" for p in missing)
        + "\n\nEjecuta primero `make preprocess` (o corre §0.3 en Colab)."
    )

if not cache.cache_is_valid(cache_dir=config.CACHE_DIR, short_side=config.CACHE_SHORT_SIDE):
    raise RuntimeError(
        f"El cache existe pero no coincide con CACHE_SHORT_SIDE={config.CACHE_SHORT_SIDE}. "
        "Regeneralo con `make cache` (o corre §0.3 en Colab)."
    )

id2label, label2id = splits.load_label_map(config.LABEL_MAP)
NUM_LABELS = len(label2id)

print("cache:", config.CACHE_DIR, "| lado corto:", config.CACHE_SHORT_SIDE)
print("clases:", NUM_LABELS)
assert NUM_LABELS == 101

cache: /content/ViT-for-101-food-app/data/interim/food-101-288 | lado corto: 288
clases: 101


## 2. DataLoaders de DeiT

`loaders.build_dataloaders("deit", ...)` lee del `AutoImageProcessor` de DeiT la resolución, la normalización (ImageNet) y el resample; no se escribe ningún número a mano.

DeiT-tiny entra a 224px, así que usa batch 32 sin gradient checkpointing (como MobileViT/ViT). La celda se adapta sola si el modelo fuera de más resolución.

In [6]:
NUM_WORKERS = min(4, os.cpu_count() or 1)  # Colab tiene 2 CPUs

TARGET_SIZE = processors.spec_for(MODEL_KEY).target_size
HEAVY = TARGET_SIZE >= 320  # alta resolucion: mas memoria por imagen

# Batch efectivo objetivo = 32, igual que MobileViT/ViT.
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if HEAVY:
        BATCH_SIZE = 8 if gpu_mem_gb >= 24 else 4  # T4: si hay OOM, baja a 2
    else:
        BATCH_SIZE = 32 if gpu_mem_gb >= 12 else 16
else:
    BATCH_SIZE = 4 if HEAVY else 8
GRAD_ACCUM = max(1, 32 // BATCH_SIZE)
GRAD_CHECKPOINT = HEAVY  # solo hace falta para modelos de alta resolucion
print("resolucion:", TARGET_SIZE, "| gradient checkpointing:", GRAD_CHECKPOINT)

dls = loaders.build_dataloaders(
    MODEL_KEY,
    train_policy="standard",
    source="cache",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    splits_to_load=("train", "val", "test"),
    images_root=config.CACHE_DIR,
    csv_path=config.TRAIN_VAL_SPLIT,
    label_map_path=config.LABEL_MAP,
    meta_dir=config.FOOD101_META_DIR,
)

train_ds = dls["train"].dataset
val_ds = dls["val"].dataset
test_ds = dls["test"].dataset

print("batch size:", BATCH_SIZE, "| grad accum:", GRAD_ACCUM, "| batch efectivo:", BATCH_SIZE * GRAD_ACCUM)
print("train:", len(train_ds), "| val:", len(val_ds), "| test:", len(test_ds))
assert len(train_ds) == 68175
assert len(val_ds) == 7575
assert len(test_ds) == 25250

batch = next(iter(dls["train"]))
print("pixel_values:", tuple(batch["pixel_values"].shape), batch["pixel_values"].dtype)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

resolucion: 224 | gradient checkpointing: False
batch size: 32 | grad accum: 1 | batch efectivo: 32
train: 68175 | val: 7575 | test: 25250
pixel_values: (32, 3, 224, 224) torch.float32


## 3. DeiT-tiny preentrenado

In [7]:
CHECKPOINT = config.MODELS[MODEL_KEY]
print("checkpoint:", CHECKPOINT)

model = training.build_model(MODEL_KEY, id2label, label2id)

params = training.count_parameters(model)
print(f"parametros totales: {params['total'] / 1e6:.2f} M")
print(f"parametros entrenables: {params['trainable'] / 1e6:.2f} M")

checkpoint: facebook/deit-tiny-patch16-224


[transformers] You passed `num_labels=101` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 23.0MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 22.9MB            

[transformers] ViTForImageClassification LOAD REPORT from: facebook/deit-tiny-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 192]) vs model:torch.Size([101, 192])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([101])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


parametros totales: 5.54 M
parametros entrenables: 5.54 M


model.safetensors: downloading bytes:           |  0.00B            

## 4. Configuración del fine-tuning

Misma receta que MobileViT, ViT y Swin (`TrainingRecipe` en `training.py`): hasta 20 épocas con early stopping (paciencia 3), mejor checkpoint por **F1 macro** en validation, warmup lineal del 5 %, `lr = 5e-4`, seed 42. El gradient checkpointing y la acumulación de gradiente se activan solos según la resolución (§2).

In [8]:
recipe = training.TrainingRecipe(batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM)

OUTPUT_DIR = config.MODELS_DIR / MODEL_KEY / "full"
RESULTS_DIR = config.REPORTS_DIR / "results" / MODEL_KEY
if IN_COLAB and USE_DRIVE:
    OUTPUT_DIR = Path(DRIVE_DIR) / "models" / MODEL_KEY / "full"
    RESULTS_DIR = Path(DRIVE_DIR) / "results" / MODEL_KEY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

training.guard_output_dir(OUTPUT_DIR, RESUME)

print("output:", OUTPUT_DIR)
print("resultados:", RESULTS_DIR)
print(training.recipe_as_dict(recipe))

output: /content/drive/MyDrive/ceia-vpc3/models/deit/full
resultados: /content/drive/MyDrive/ceia-vpc3/results/deit
{'epochs': 20, 'learning_rate': 0.0005, 'weight_decay': 0.01, 'warmup_ratio': 0.05, 'batch_size': 32, 'grad_accum_steps': 1, 'early_stopping_patience': 3, 'metric_for_best': 'f1_macro', 'seed': 42}


## 5. Trainer

In [9]:
trainer = training.build_trainer(
    model,
    recipe,
    output_dir=OUTPUT_DIR,
    train_dataset=train_ds,
    val_dataset=val_ds,
    num_workers=NUM_WORKERS,
    gradient_checkpointing=GRAD_CHECKPOINT,
)

## 6. Fine-tuning completo

In [10]:
elapsed_train = training.run_training(trainer, resume=RESUME)
print(f"tiempo total: {elapsed_train / 3600:.2f} h")
print("best F1 macro:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,2.110897,2.115312,0.461782,0.460041,0.460041
2,1.751319,1.778193,0.540066,0.546106,0.546106
3,1.510634,1.572791,0.589043,0.592007,0.592007
4,1.330036,1.453585,0.626667,0.625308,0.625308
5,1.173978,1.386241,0.639076,0.641572,0.641572
6,1.021659,1.396278,0.642376,0.644095,0.644095
7,0.925750,1.304987,0.659142,0.657626,0.657626
8,0.774011,1.278800,0.676964,0.675909,0.675909


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 7. Historial por época

In [11]:
history = pd.DataFrame(trainer.state.log_history)
epoch_metrics = history[history["eval_loss"].notna()][
    ["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro", "eval_f1_weighted"]
].copy()
epoch_metrics

,epoch,eval_loss,eval_accuracy,eval_f1_macro,eval_f1_weighted
10,1.0,2.115312,0.461782,0.460041,0.460041
22,2.0,1.778193,0.540066,0.546106,0.546106
33,3.0,1.572791,0.589043,0.592007,0.592007
45,4.0,1.453585,0.626667,0.625308,0.625308
57,5.0,1.386241,0.639076,0.641572,0.641572
68,6.0,1.396278,0.642376,0.644095,0.644095
80,7.0,1.304987,0.659142,0.657626,0.657626
92,8.0,1.278800,0.676964,0.675909,0.675909


## 8. Guardar el mejor modelo

In [12]:
BEST_DIR = training.save_best_model(trainer, OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-09-25 22:59:34.519 | SUCCESS  | vit_for_101_food_app.modeling.training:save_best_model:177 - mejor modelo guardado en /content/drive/MyDrive/ceia-vpc3/models/deit/full/best


## 8.1 Descargar el modelo a tu computadora

Comprime el mejor modelo y, en Colab, lo descarga al navegador. El `.zip` trae los pesos (`model.safetensors`) y la config, listos para `AutoModelForImageClassification.from_pretrained(carpeta_descomprimida)`.

Con `USE_DRIVE = True` el modelo también queda en tu Drive (`OUTPUT_DIR/best`), por si la descarga del navegador falla.

In [13]:
zip_path = training.zip_model(BEST_DIR, Path("/content") / f"{MODEL_KEY}-best" if IN_COLAB else OUTPUT_DIR / f"{MODEL_KEY}-best")
training.download_to_browser(zip_path)

2026-09-25 22:59:37.632 | SUCCESS  | vit_for_101_food_app.modeling.training:zip_model:185 - modelo comprimido en /content/deit-best.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Evaluación final sobre test

`test` se usa recién acá, una sola vez. Las predicciones se guardan **por imagen** (`predictions_test.csv`): con eso se recalcula cualquier métrica y se compara contra los otros modelos sin reentrenar.

In [14]:
import time

t0 = time.time()
test_output = trainer.predict(test_ds)
elapsed_test = time.time() - t0

test_frame = splits.load_split("test", csv_path=config.TRAIN_VAL_SPLIT, meta_dir=config.FOOD101_META_DIR)
test_preds = evaluation.predictions_frame(test_frame, test_output.predictions, id2label)
test_preds.to_csv(RESULTS_DIR / "predictions_test.csv", index=False)

test_metrics = evaluation.split_metrics(test_preds)
print(f"tiempo test: {elapsed_test:.1f} s")
pd.Series(test_metrics)

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,2.110897,2.115312,0.461782,0.460041,0.460041
2,1.751319,1.778193,0.540066,0.546106,0.546106
3,1.510634,1.572791,0.589043,0.592007,0.592007
4,1.330036,1.453585,0.626667,0.625308,0.625308
5,1.173978,1.386241,0.639076,0.641572,0.641572
6,1.021659,1.396278,0.642376,0.644095,0.644095
7,0.925750,1.304987,0.659142,0.657626,0.657626
8,0.774011,1.278800,0.676964,0.675909,0.675909


tiempo test: 99.0 s


,0
n,25250.000000
n_clases,101.000000
accuracy,0.731366
top5_accuracy,0.918495
f1_macro,0.731761
f1_weighted,0.731761


## 10. Subset del benchmark por tercil de dificultad

La tabla que contesta la pregunta del proyecto. El subset es **el mismo archivo para todos los modelos** y `load_benchmark_subset` verifica su `sha256` antes de usarlo.

In [15]:
subset = evaluation.load_benchmark_subset()
benchmark = evaluation.benchmark_metrics(test_preds, subset)
benchmark.to_csv(RESULTS_DIR / "benchmark_por_tercil.csv")
benchmark

,n,n_clases,accuracy,top5_accuracy,f1_macro,f1_weighted
grupo,,,,,,
subset,2525,101,0.744554,0.927525,0.744861,0.744861
facil,850,34,0.831765,0.962353,0.879906,0.879906
medio,825,33,0.753939,0.926061,0.819088,0.819088
dificil,850,34,0.648235,0.894118,0.704811,0.704811


## 11. Reporte por clase

In [16]:
class_names = [id2label[i] for i in range(NUM_LABELS)]
report = classification_report(
    test_preds["label_id"], test_preds["pred_id"],
    labels=list(range(NUM_LABELS)), target_names=class_names,
    output_dict=True, zero_division=0,
)
report_df = pd.DataFrame(report).T.loc[class_names].sort_values("f1-score")
report_df.to_csv(RESULTS_DIR / "report_por_clase_test.csv")

print("10 clases con peor F1")
display(report_df.head(10))
print("10 clases con mejor F1")
display(report_df.tail(10))

10 clases con peor F1


,precision,recall,f1-score,support
apple_pie,0.555556,0.320,0.406091,250.0
foie_gras,0.588957,0.384,0.464891,250.0
pork_chop,0.433544,0.548,0.484099,250.0
steak,0.430769,0.560,0.486957,250.0
filet_mignon,0.593407,0.432,0.500000,250.0
bread_pudding,0.486111,0.560,0.520446,250.0
ravioli,0.525362,0.580,0.551331,250.0
breakfast_burrito,0.578059,0.548,0.562628,250.0
tuna_tartare,0.462121,0.732,0.566563,250.0
chocolate_mousse,0.536585,0.616,0.573557,250.0


10 clases con mejor F1


,precision,recall,f1-score,support
mussels,0.917031,0.840,0.876827,250.0
spaghetti_bolognese,0.896266,0.864,0.879837,250.0
spaghetti_carbonara,0.850746,0.912,0.880309,250.0
hot_and_sour_soup,0.877953,0.892,0.884921,250.0
pho,0.879377,0.904,0.891519,250.0
bibimbap,0.908714,0.876,0.892057,250.0
macarons,0.942478,0.852,0.894958,250.0
miso_soup,0.927350,0.868,0.896694,250.0
oysters,0.893701,0.908,0.900794,250.0
edamame,0.979757,0.968,0.973843,250.0


## 12. Confusiones más frecuentes

In [17]:
errores = test_preds[~test_preds["correct"]]
confusiones = (
    errores.groupby(["class_dir", "pred_class"]).size()
    .rename("n").sort_values(ascending=False).reset_index()
)
confusiones.to_csv(RESULTS_DIR / "confusiones_test.csv", index=False)
print("errores totales:", len(errores), "de", len(test_preds))
confusiones.head(15)

errores totales: 6783 de 25250


,class_dir,pred_class,n
0,filet_mignon,steak,55
1,beef_tartare,tuna_tartare,43
2,chocolate_cake,chocolate_mousse,36
3,apple_pie,bread_pudding,34
4,frozen_yogurt,ice_cream,34
5,steak,filet_mignon,30
6,baby_back_ribs,pork_chop,30
7,prime_rib,steak,29
8,falafel,crab_cakes,27
9,fried_rice,risotto,26


## 13. Costo arquitectónico: parámetros, FLOPs y latencia

Se mide con `modeling/evaluation.py`, el mismo código que los otros modelos: esa es la condición para comparar arquitecturas. Latencia con batch 1 (una foto por vez), mediana y p90 de 100 corridas, en GPU y CPU.

In [18]:
device = trainer.args.device
sample = test_ds[0]["pixel_values"].unsqueeze(0)

n_params = sum(p.numel() for p in model.parameters())
costo = {
    "params_m": n_params / 1e6,
    "size_mb_fp32": n_params * 4 / 1024**2,
    **evaluation.count_flops(model, sample.to(device)),
}
latencia = {
    "gpu": evaluation.measure_latency(model, sample, device) if device.type == "cuda" else None,
    "cpu": evaluation.measure_latency(model, sample, "cpu"),
}
model.to(device)

print(pd.Series(costo))
pd.DataFrame({k: v for k, v in latencia.items() if v is not None})

params_m         5.543909
size_mb_fp32    21.148335
gflops           2.507021
gmacs            1.253511
dtype: float64


,gpu,cpu
device,Tesla T4,cpu
batch_size,1,1
cpu_threads,1,1
n_runs,100,100
median_ms,10.460598,38.14995
p90_ms,11.343895,55.858681
mean_ms,10.783302,43.10558


## 14. Guardar métricas

In [21]:
import json

metrics = {
    "model_key": MODEL_KEY,
    "checkpoint": CHECKPOINT,
    "receta": {
        **training.recipe_as_dict(recipe),
        "epochs_entrenadas": trainer.state.epoch,
    },
    "tiempo_entrenamiento_s": elapsed_train,
    "best_val_f1_macro": trainer.state.best_metric,
    "test": test_metrics,
    "benchmark_por_tercil": benchmark.to_dict(orient="index"),
    "costo": costo,
    "latencia": latencia,
    "entorno": training.software_versions(),
}
(RESULTS_DIR / "metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("guardado en", RESULTS_DIR)
for p in sorted(RESULTS_DIR.iterdir()):
    print(" -", p.name)

guardado en /content/drive/MyDrive/ceia-vpc3/results/deit
 - benchmark_por_tercil.csv
 - confusiones_test.csv
 - metrics.json
 - predictions_test.csv
 - report_por_clase_test.csv


## Resumen

- Todo el código de modelo/entrenamiento vive en `vit_for_101_food_app/modeling/training.py`; esta notebook solo orquesta (idéntica a `3.2-swin` salvo `MODEL_KEY`).
- `train` para entrenar, `val` para early stopping y selección de checkpoint, `test` una sola vez.
- La métrica que contesta la pregunta del proyecto es la del **subset por tercil** (§10).
- El mejor modelo queda en `models/deit/full/best/` y se descarga a tu compu (§8.1); **no** se versiona.

DeiT-tiny es el punto más liviano de la familia ViT del benchmark. Con los `metrics.json` de DeiT-tiny, Swin-tiny, ViT-base y MobileViT se arma la frontera de eficiencia (accuracy vs params / vs latencia) sin reentrenar.